<a href="https://colab.research.google.com/github/AjMing/BigData/blob/main/Streaming/pyspark_streaming_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PySpark Streaming


In [1]:
# Install PySpark
!pip install pyspark -q

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import time

# Create Spark Session
spark = SparkSession.builder \
    .appName("ColabStreaming") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "2") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

Spark version: 4.0.2


## Example 1: Rate Source Stream (Auto-generates data)
Generates data automatically without external sources

In [9]:
# Create streaming DataFrame using rate source
# Generates rows with timestamp and value columns at specified rate
rate_stream = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 5) \
    .load()

# Add some processing
processed = rate_stream.select(
    col("timestamp"),
    col("value"),
    (col("value") % 10).alias("category")
)

# Count by category
counts = processed.groupBy("category").count()

# Start streaming query (runs for 20 seconds)
query = counts.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("rate_counts") \
    .start()

# Let it run for 20 seconds
time.sleep(20)

# Display results
spark.sql("SELECT * FROM rate_counts ORDER BY category").show()

# Stop the query
query.stop()

+--------+-----+
|category|count|
+--------+-----+
|       0|   10|
|       1|   10|
|       2|   10|
|       3|   10|
|       4|   10|
|       5|    9|
|       6|    9|
|       7|    9|
|       8|    9|
|       9|    9|
+--------+-----+



## Example 2: File-based Streaming
Monitor a directory for new CSV files

In [4]:
import os

# Create directories
input_dir = "/tmp/stream_input"
checkpoint_dir = "/tmp/checkpoint"
output_dir = "/tmp/stream_output"

os.makedirs(input_dir, exist_ok=True)
os.makedirs(checkpoint_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

# Define schema
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("value", IntegerType(), True),
    StructField("category", StringType(), True)
])

# Create streaming DataFrame
file_stream = spark.readStream \
    .format("csv") \
    .option("header", "true") \
    .schema(schema) \
    .load(input_dir)

# Process: aggregate by category
aggregated = file_stream.groupBy("category") \
    .agg(
        count("*").alias("count"),
        avg("value").alias("avg_value"),
        sum("value").alias("sum_value")
    )

# Write to memory for querying
query2 = aggregated.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("file_stream_agg") \
    .option("checkpointLocation", checkpoint_dir) \
    .start()

print("Streaming query started. Waiting for files...")

Streaming query started. Waiting for files...


In [5]:
# Generate sample CSV files
import pandas as pd
import random

categories = ['A', 'B', 'C', 'D']

# Create file 1
df1 = pd.DataFrame({
    'id': range(1, 11),
    'name': [f'Item_{i}' for i in range(1, 11)],
    'value': [random.randint(10, 100) for _ in range(10)],
    'category': [random.choice(categories) for _ in range(10)]
})
df1.to_csv(f"{input_dir}/data_1.csv", index=False)
print("Created data_1.csv")

time.sleep(3)

# Check results
spark.sql("SELECT * FROM file_stream_agg ORDER BY category").show()

# Create file 2
df2 = pd.DataFrame({
    'id': range(11, 21),
    'name': [f'Item_{i}' for i in range(11, 21)],
    'value': [random.randint(10, 100) for _ in range(10)],
    'category': [random.choice(categories) for _ in range(10)]
})
df2.to_csv(f"{input_dir}/data_2.csv", index=False)
print("\nCreated data_2.csv")

time.sleep(3)

# Check updated results
print("\nUpdated results:")
spark.sql("SELECT * FROM file_stream_agg ORDER BY category").show()

# Stop query
query2.stop()

Created data_1.csv
+--------+-----+-----------------+---------+
|category|count|        avg_value|sum_value|
+--------+-----+-----------------+---------+
|       B|    5|             70.2|      351|
|       C|    2|             73.0|      146|
|       D|    3|65.66666666666667|      197|
+--------+-----+-----------------+---------+


Created data_2.csv

Updated results:
+--------+-----+------------------+---------+
|category|count|         avg_value|sum_value|
+--------+-----+------------------+---------+
|       A|    3|45.666666666666664|      137|
|       B|    7| 67.42857142857143|      472|
|       C|    6|46.833333333333336|      281|
|       D|    4|             52.25|      209|
+--------+-----+------------------+---------+



## Example 3: Windowed Aggregations with Rate Source

In [6]:
# Create rate stream with faster generation
rate_stream = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 10) \
    .load()

# Add derived columns
enriched = rate_stream.select(
    col("timestamp"),
    col("value"),
    (col("value") % 5).alias("group")
)

# Windowed aggregation: 10 second windows, sliding every 5 seconds
windowed = enriched \
    .groupBy(
        window(col("timestamp"), "10 seconds", "5 seconds"),
        col("group")
    ) \
    .agg(
        count("*").alias("count"),
        min("value").alias("min_value"),
        max("value").alias("max_value")
    )

# Start query
query3 = windowed.writeStream \
    .outputMode("update") \
    .format("memory") \
    .queryName("windowed_agg") \
    .start()

# Let it run
time.sleep(25)

# Display results
result = spark.sql("""
    SELECT
        window.start,
        window.end,
        `group`,
        count,
        min_value,
        max_value
    FROM windowed_agg
    ORDER BY window.start DESC, `group`
    LIMIT 20
""")

result.show(truncate=False)

query3.stop()

+-------------------+-------------------+-----+-----+---------+---------+
|start              |end                |group|count|min_value|max_value|
+-------------------+-------------------+-----+-----+---------+---------+
|2026-04-16 22:51:30|2026-04-16 22:51:40|0    |1    |195      |195      |
|2026-04-16 22:51:30|2026-04-16 22:51:40|0    |3    |195      |205      |
|2026-04-16 22:51:30|2026-04-16 22:51:40|0    |5    |195      |215      |
|2026-04-16 22:51:30|2026-04-16 22:51:40|0    |7    |195      |225      |
|2026-04-16 22:51:30|2026-04-16 22:51:40|0    |9    |195      |235      |
|2026-04-16 22:51:30|2026-04-16 22:51:40|1    |1    |196      |196      |
|2026-04-16 22:51:30|2026-04-16 22:51:40|1    |3    |196      |206      |
|2026-04-16 22:51:30|2026-04-16 22:51:40|1    |5    |196      |216      |
|2026-04-16 22:51:30|2026-04-16 22:51:40|1    |7    |196      |226      |
|2026-04-16 22:51:30|2026-04-16 22:51:40|1    |9    |196      |236      |
|2026-04-16 22:51:30|2026-04-16 22:51:

## Example 4: Simulated Real-time Word Count

In [7]:
# Create a directory for text files
text_input_dir = "/tmp/text_stream"
os.makedirs(text_input_dir, exist_ok=True)

# Read text files as stream
text_stream = spark.readStream \
    .format("text") \
    .load(text_input_dir)

# Word count processing
words = text_stream.select(
    explode(split(col("value"), " ")).alias("word")
)

# Filter empty words and count
word_counts = words \
    .filter(col("word") != "") \
    .groupBy("word") \
    .count() \
    .orderBy(desc("count"))

# Start streaming
query4 = word_counts.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("word_counts") \
    .start()

print("Word count streaming started...")

Word count streaming started...


In [8]:
# Generate text files
texts = [
    "apache spark streaming is powerful for real-time data processing",
    "spark streaming enables scalable high-throughput fault-tolerant stream processing",
    "pyspark brings the power of spark to python developers",
    "real-time analytics with spark streaming in google colab"
]

for i, text in enumerate(texts):
    with open(f"{text_input_dir}/text_{i}.txt", "w") as f:
        f.write(text)
    print(f"Created text_{i}.txt")
    time.sleep(2)

    # Show current word counts
    print(f"\nWord counts after file {i+1}:")
    spark.sql("SELECT * FROM word_counts LIMIT 10").show()

# Final results
print("\n=== Final Top 15 Words ===")
spark.sql("SELECT * FROM word_counts LIMIT 15").show()

query4.stop()

Created text_0.txt

Word counts after file 1:
+----------+-----+
|      word|count|
+----------+-----+
| real-time|    1|
|       for|    1|
|  powerful|    1|
|     spark|    1|
|      data|    1|
|        is|    1|
|    apache|    1|
| streaming|    1|
|processing|    1|
+----------+-----+

Created text_1.txt

Word counts after file 2:
+----------+-----+
|      word|count|
+----------+-----+
|     spark|    2|
| streaming|    2|
|processing|    2|
|    stream|    1|
| real-time|    1|
|       for|    1|
|  scalable|    1|
|  powerful|    1|
|      data|    1|
|        is|    1|
+----------+-----+

Created text_2.txt

Word counts after file 3:
+----------+-----+
|      word|count|
+----------+-----+
|     spark|    3|
| streaming|    2|
|processing|    2|
|        to|    1|
|    stream|    1|
| real-time|    1|
|       for|    1|
|    brings|    1|
|       the|    1|
|  scalable|    1|
+----------+-----+

Created text_3.txt

Word counts after file 4:
+----------+-----+
|      word|cou

## Example 5: Monitoring Streaming Queries

In [ ]:
# Start a simple streaming query
monitoring_stream = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 100) \
    .load()

query5 = monitoring_stream \
    .groupBy((col("value") % 10).alias("bucket")) \
    .count() \
    .writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("monitoring") \
    .start()

# Monitor the query
time.sleep(5)

print("=== Query Status ===")
print(f"Query ID: {query5.id}")
print(f"Query Name: {query5.name}")
print(f"Is Active: {query5.isActive}")
print(f"\nLast Progress:")
print(query5.lastProgress)

# Get all active streams
print("\n=== All Active Streams ===")
for s in spark.streams.active:
    print(f"- {s.name} (ID: {s.id})")

query5.stop()

In [ ]:
# Clean up: Stop all streaming queries
for stream in spark.streams.active:
    print(f"Stopping {stream.name}...")
    stream.stop()

print("\nAll streaming queries stopped.")

# Stop Spark session (optional)
# spark.stop()